In [1]:
import numpy as np
import pandas as pd
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords
import re
import ftfy
import html
pd.set_option('display.max_colwidth', None)

In [2]:
df=pd.read_csv("development.csv",delimiter=",", index_col="Id")

### *Source* feature inspection

In [3]:
#TODO:cambiare le print in inglese, in futuro dovrò fare catboost/target encoding
n_nan = df['source'].isna().sum()
n_placeholder = (df['source'] == '\\N').sum()
n_empty = (df['source'].astype(str).str.strip() == '').sum()

print(f"NaN: {n_nan}")
print(f"Placeholder \\N: {n_placeholder}")
print(f"Stringhe vuote: {n_empty}")

df['source'] = df['source'].replace('\\N', np.nan)
df['source'] = df['source'].replace('', np.nan)
df['source'] = df['source'].fillna('Other')

min_freq = 5  
source_counts = df['source'].value_counts()
sources_kept = source_counts[source_counts >= min_freq].index.tolist()

df['source'] = np.where(df['source'].isin(sources_kept),df['source'],'Other')

final_source_counts = df['source'].value_counts()

print(f"Numero categorie finali (incluso Other): {len(final_source_counts)}")

NaN: 0
Placeholder \N: 294
Stringhe vuote: 0
Numero categorie finali (incluso Other): 489


### *Title* feature inspection

In [4]:
n_nan_title = df['title'].isna().sum()
n_placeholders_title = (df['title']== '\\N').sum()
n_empty_title = (df['title'].astype(str).str.strip()=='').sum()

print(f"Number of NaN rows: {n_nan_title}")
print(f"Number of placeholders (\\N): {n_placeholders_title}")
print(f"Number of empty rows: {n_empty_title}")
print("Titles Sample:")
print(df['title'].sample(10))

Number of NaN rows: 1
Number of placeholders (\N): 0
Number of empty rows: 2
Titles Sample:
Id
13597                         Mare&#39;s 51-yard field goal lifts Dolphins
53235                          Gay Marriage Amendment Not Coming Soon (AP)
52110                           Veracode debuts system to test source code
51411                      Gallery: MacBook Air Makes Its Slim, Sexy Debut
59805                               Plan May Keep Bird Off Endangered List
10109              4-goal surge powers Ducks past Predators \\n    (AP)\\n
4762                                   Scarpato is now center of attention
69104                                    UK challenge of India outsourcing
20465    Norton's "Canary" technology creates "vulnerability signatures"  
7886                              Health Savings Accounts for Poor Tested 
Name: title, dtype: object


### *Article* feature inspection

In [5]:
n_nan_article = df['article'].isna().sum()
n_placeholders_article = (df['article']=='\\N').sum()
n_empty_article = (df['article'].astype(str).str.strip()=='').sum()

print(f"Number of NaN rows: {n_nan_article}")
print(f"Number of placeholders (\\N): {n_placeholders_article}")
print(f"Number of empty rows: {n_empty_article}")
print("Articles Sample")
print(df['article'].sample(10))

Number of NaN rows: 1
Number of placeholders (\N): 1874
Number of empty rows: 7
Articles Sample
Id
76616                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            The man charged with murdering the British backpacker Peter Falconio will face trial next year, a magistrate in the Australian city of Darwin ruled last night. 
54893                                                                                                                                                                                                                              

### *PageRank* feature inspection

In [6]:
n_nan_pr = df['page_rank'].isna().sum()
n_placeholders_pr = (df['page_rank']=='\\N').sum()
n_empty_pr = (df['page_rank'].astype(str).str.strip()=='').sum()
rank_5=np.array([df['page_rank']==5]).sum()

print(f"Number of NaN rows: {n_nan_pr}")
print(f"Number of placeholders (\\N): {n_placeholders_pr}")
print(f"Number of empty rows: {n_empty_pr}")
print(f"Number of articles with PageRank 5: {rank_5}")

Number of NaN rows: 0
Number of placeholders (\N): 0
Number of empty rows: 0
Number of articles with PageRank 5: 73891


### *Timestamp* feature inspection 

In [7]:
n_nan_time = df['timestamp'].isna().sum()
n_placeholders_time = (df['timestamp']=='\\N').sum()
n_empty_time = (df['timestamp'].astype(str).str.strip()=='').sum()
n_uslesess_time=np.array([df['timestamp']=="0000-00-00 00:00:00"]).sum()

print(f"Number of NaN rows: {n_nan_time}")
print(f"Number of placeholders (\\N): {n_placeholders_time}")
print(f"Number of empty rows: {n_empty_time}")
print(f"Number invalid dates (0000-00-00 00:00:00): {n_uslesess_time}")
print(df['timestamp'].sample(10))

Number of NaN rows: 0
Number of placeholders (\N): 0
Number of empty rows: 0
Number invalid dates (0000-00-00 00:00:00): 27750
Id
34576    2008-01-29 21:01:40
47418    2004-12-22 16:10:51
13520    0000-00-00 00:00:00
67628    2005-01-03 19:54:29
74205    2004-12-10 06:39:57
22421    0000-00-00 00:00:00
18049    0000-00-00 00:00:00
32934    2007-03-13 10:05:26
76642    0000-00-00 00:00:00
30159    0000-00-00 00:00:00
Name: timestamp, dtype: object


### *Timestamp* feature processing

In [8]:
def process_timestamp(df):
    df = df.copy()

    dt = pd.to_datetime(df['timestamp'], errors='coerce')

    df['has_date'] = dt.notna().astype(int)
    df['quarter'] = dt.dt.quarter.fillna(-1).astype(int)
    df['is_weekend'] = dt.dt.dayofweek.isin([5, 6]).fillna(False).astype(int)

    df = df.drop(columns=['timestamp'])
    return df

df = process_timestamp(df)

timestamp_cols = ['has_date', 'is_weekend','quarter']
print(f"Timestamp expanded columns: {timestamp_cols}")
print("'timestamp' raw column deleted")
print("Sample of 10 timestamps:")
print(df[timestamp_cols].sample(10))

Timestamp expanded columns: ['has_date', 'is_weekend', 'quarter']
'timestamp' raw column deleted
Sample of 10 timestamps:
       has_date  is_weekend  quarter
Id                                  
12535         1           0        3
48873         1           0        4
17429         1           1        2
4496          0           0       -1
62258         1           0        4
6540          0           0       -1
56561         0           0       -1
48580         1           0        3
68108         1           0        4
58346         0           0       -1


### *Title* feature stemming

In [9]:
import re
import html
import ftfy
import pandas as pd

def clean_text_light(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = ftfy.fix_text(text)

    text = html.unescape(text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\,\-\%\$\€\£]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def process_title_final(df):
    df = df.copy()

    df['title_clean'] = df['title'].apply(clean_text_light)
    df = df.drop(columns=['title'])

    return df


df = process_title_final(df)

print("Sample of 10 cleaned titles:")
print(df['title_clean'].sample(10).to_string(index=False))


Sample of 10 cleaned titles:
Id
            nikkei up at midday, led by telecoms reuters
        cleric rushes to see arafat burial plans proceed
                                      parma sack baldini
entourage star grenier unravels personal mystery reuters
         fahrenheit 9 11 gets election-eve pay-tv airing
          europe, islam s new front line the netherlands
                trusecure, betrusted to merge and rename
                                           looking ahead
                gagne blows it, but rangers win in 10 ap
                    pharmaceutical sector mixed on merck


### *Article* feature stemming

In [ ]:
import re
import html
import ftfy
import pandas as pd
##TODO remove comments and change variables, also for title above
def clean_text_light(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Fix broken encodings
    text = ftfy.fix_text(text)

    # Decode HTML entities
    text = html.unescape(text)

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove common web/markup leftovers (keep it simple)
    text = re.sub(r'\b(a\s+href|href|img\s+src|nbsp|read\s+more|click\s+here)\b',' ',text,flags=re.IGNORECASE)


    # Lowercase
    text = text.lower()

    # Keep letters, numbers, and useful punctuation for n-grams
    text = re.sub(r"[^a-z0-9\s\.\,\-\%\$\€\£]", " ", text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def process_article_final(df):
    df = df.copy()

    # Normalize placeholders / missing
    df['article'] = df['article'].replace('\\N', '').fillna('').astype(str)

    df['article_clean'] = df['article'].apply(clean_text_light)
    df = df.drop(columns=['article'])

    return df


# === APPLY ===
df = process_article_final(df)

# === DEBUG / SANITY CHECK ===
print("Sample of 10 cleaned articles:")
print(df['article_clean'].sample(10).to_string(index=False))

Sample of 10 cleaned articles:
Id
                                                  ap - new mississippi coach ed orgeron was charged with repeated domestic violence more than a decade ago when he was an assistant at miami, according to records obtained friday by the associated press.
                                                                                                                              no country for old men received another major honor when it was selected best feature film by the producers guild of america.
                                                       the washington capitals collective confidence received a considerable boost after tuesday s drubbing of the ottawa senators, their second win over the eastern conference s best team in three days.
                                                                          moscow reuters - russia s government approved the kyoto protocol on thursday and sent the climate change pact to the state duma, the low

In [14]:
df['text'] = (df['title_clean'] + ' ' + df['title_clean'] + ' ' + df['article_clean']).str.strip()

# opzionale: se vuoi ridurre memoria dopo aver creato text
df = df.drop(columns=['title_clean', 'article_clean'])


### Encoding categorical features

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from category_encoders.cat_boost import CatBoostEncoder

# ----------------------------
# Columns
# ----------------------------
TEXT_COL = 'text'
CAT_COLS = ['source']
NUM_COLS = ['page_rank', 'has_date', 'is_weekend', 'quarter']
TARGET_COL = 'label'

X = df[[TEXT_COL] + CAT_COLS + NUM_COLS]
y = df[TARGET_COL]

# ----------------------------
# Train / validation split
# ----------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ----------------------------
# Preprocessing
# ----------------------------
preprocess = ColumnTransformer(
    transformers=[
        (
            'tfidf',
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=5,
                max_df=0.95,
                max_features=150_000
            ),
            TEXT_COL
        ),
        (
            'catboost',
            CatBoostEncoder(
                cols=CAT_COLS,
                a=1.0,
                random_state=42
            ),
            CAT_COLS
        ),
        (
            'num',
            'passthrough',
            NUM_COLS
        )
    ]
)

# ----------------------------
# Fit preprocessing only (sanity check)
# ----------------------------
X_train_trans = preprocess.fit_transform(X_train, y_train)
X_val_trans = preprocess.transform(X_val)

print("Train shape:", X_train_trans.shape)
print("Validation shape:", X_val_trans.shape)


Train shape: (63997, 82556)
Validation shape: (16000, 82556)


In [18]:
import lightgbm as lgb
from sklearn.metrics import f1_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ----------------------------
# Class weights (important!)
# ----------------------------
classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, class_weights))


lgb_clf = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(classes),
    class_weight=class_weight_dict,
    n_estimators=200,          # ↓↓↓
    learning_rate=0.1,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.6,     # ↓↓↓ CRITICO
    random_state=42,
    n_jobs=-1,
    verbose=-1                # toglie spam
)

lgb_clf.fit(X_train_trans, y_train)

y_val_pred = lgb_clf.predict(X_val_trans)
print("Validation Macro F1:",f1_score(y_val, y_val_pred, average='macro'))


/Users/giorgiozoccatelli/miniforge3/envs/data/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Validation Macro F1: 0.6848846672233988


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score
)

# Label mapping (dal pdf)
label_names = {
    0: "International News",
    1: "Business",
    2: "Technology",
    3: "Entertainment",
    4: "Sports",
    5: "General News",
    6: "Health"
}

# ---- 1) Metriche globali
macro_f1 = f1_score(y_val, y_val_pred, average='macro')
micro_f1 = f1_score(y_val, y_val_pred, average='micro')
weighted_f1 = f1_score(y_val, y_val_pred, average='weighted')

print("=== GLOBAL METRICS ===")
print(f"Macro F1    : {macro_f1:.6f}")
print(f"Micro F1    : {micro_f1:.6f}")
print(f"Weighted F1 : {weighted_f1:.6f}")
print()

# ---- 2) Classification report per classe
target_names = [label_names[i] for i in sorted(label_names.keys())]

print("=== CLASSIFICATION REPORT ===")
print(classification_report(
    y_val,
    y_val_pred,
    labels=sorted(label_names.keys()),
    target_names=target_names,
    digits=4
))

# ---- 3) Confusion matrix come DataFrame leggibile
cm = confusion_matrix(y_val, y_val_pred, labels=sorted(label_names.keys()))
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)

print("=== CONFUSION MATRIX (counts) ===")
display(cm_df)

# ---- 4) Confusion matrix normalizzata per riga (recall per classe)
cm_norm = cm / cm.sum(axis=1, keepdims=True)
cm_norm_df = pd.DataFrame(cm_norm, index=target_names, columns=target_names)

print("=== CONFUSION MATRIX (row-normalized: recall profile) ===")
display(cm_norm_df.round(3))

# ---- 5) Top confusion pairs (dove sbaglia di più)
off_diag = cm.copy()
np.fill_diagonal(off_diag, 0)

pairs = []
for i, true_name in enumerate(target_names):
    for j, pred_name in enumerate(target_names):
        if i != j and off_diag[i, j] > 0:
            pairs.append((off_diag[i, j], true_name, pred_name))

pairs.sort(reverse=True, key=lambda x: x[0])

print("=== TOP 15 CONFUSIONS (True -> Pred) ===")
for k, (cnt, t, p) in enumerate(pairs[:15], start=1):
    print(f"{k:02d}. {t} -> {p}: {cnt}")
